# Week 1 · Day 2 — MLflow Tracking Server, MLproject & First Experiment

---

## 🔁 Day 1 Recap

- Python 3.11 + `geo-mro` conda env ✅
- Full project directory scaffold ✅
- `src/config/settings.py` + `src/utils/logger.py` ✅
- GitHub repo, `main` + `develop` branches, CI pipeline ✅
- All critical imports smoke-tested ✅

---

## 🎯 Day 2 Objective

Stand up the **MLflow experiment tracking system** so every future model run is automatically logged, versioned, and comparable.

| Step | Action | Output |
|------|--------|--------|
| 1 | Configure MLflow tracking URI (SQLite backend) | `mlflow.db` persisted |
| 2 | Create experiment taxonomy (one per module) | 5 named experiments |
| 3 | Write `MLproject` YAML + entry points | `mlflow run .` works |
| 4 | Log a structured dummy experiment | Params + metrics + artifacts |
| 5 | Register a model in the Model Registry | v1 in `Staging` |
| 6 | Verify everything in the MLflow UI | `localhost:5000` live |

---

## ❓ Why MLflow Before Any Data Work?

Every downstream module produces a model or a scored artefact:
- W3 → ABC/VED/FNS classifier (weights, thresholds)
- W6 → Croston's forecast (alpha, beta)
- W8 → Newsvendor Q* (service level, demand params)
- W10 → Nash equilibrium (payoff matrices, strategies)

Without tracking: you can't answer *"which threshold gave me the best Ci distribution?"*  
With MLflow from Day 2: every experiment is reproducible and comparable from day one.

---

## ⚙️ Architecture (end-state of Day 2)

```
geo-aware-mro/
├─ MLproject                    ← entry points for mlflow run
├─ mlflow/
│   ├─ mlruns/                  ← experiment artefacts (DVC-ignored)
│   └─ mlflow.db                ← SQLite backend (DVC-ignored)
├─ src/
│   └─ mlflow_setup/
│       ├─ __init__.py
│       ├─ experiments.py       ← experiment registry + helper
│       └─ dummy_run.py         ← runnable entry point
```

---
## STEP 0 — Path setup & Day 1 gate

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

# ── Resolve project root (same pattern as Day 1) ──────────────────────────
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    cwd = Path.cwd()
    ROOT = cwd
    for parent in [cwd] + list(cwd.parents):
        if (parent / ".git").exists() or (parent / "environment.yml").exists():
            ROOT = parent
            break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# ── Import Day 1 artefacts ────────────────────────────────────────────────
from src.config.settings import settings
from src.utils.logger import get_logger

logger = get_logger("Day2")

# ── Gate: confirm Day 1 outputs exist ─────────────────────────────────────
DAY1_GATES = [
    ROOT / "src" / "config" / "settings.py",
    ROOT / "src" / "utils" / "logger.py",
    ROOT / "environment.yml",
    ROOT / ".gitignore",
]
for gate in DAY1_GATES:
    assert gate.exists(), f"❌ Day 1 output missing: {gate}  →  Run W1_D1 first"

logger.info(f"📁 ROOT          : {ROOT}")
logger.info(f"📦 PROCESSED_DIR : {settings.PROCESSED_DIR}")
logger.info(f"🗄️  MLFLOW_URI    : {settings.MLFLOW_URI}")
print("\n✅ Day 1 gates passed — proceeding to Day 2")

---
## STEP 1 — Configure MLflow tracking server

We use **SQLite** as the backend store — zero infrastructure, fully self-contained,
sufficient for a 12-week solo project. The URI is defined in `settings.py`
so it's the same whether you run from notebook, CLI, or Docker.

In [ ]:
import mlflow

# ── Point MLflow at our SQLite backend ───────────────────────────────────
# settings.MLFLOW_URI  =  "sqlite:////<ROOT>/mlflow/mlruns/mlflow.db"
mlflow.set_tracking_uri(settings.MLFLOW_URI)

# ── Verify the connection ────────────────────────────────────────────────
client = mlflow.tracking.MlflowClient()
tracking_uri = mlflow.get_tracking_uri()

logger.info(f"✅ MLflow tracking URI : {tracking_uri}")
logger.info(f"   MLflow version      : {mlflow.__version__}")

# ── Confirm the db file will be created in the right place ───────────────
db_path = settings.MLFLOW_DIR / "mlflow.db"
print(f"\n📍 SQLite backend will be at:")
print(f"   {db_path}")
print(f"   Exists now: {db_path.exists()}")
print(f"   (It is created automatically on first experiment write)")

---
## STEP 2 — Write `src/mlflow_setup/experiments.py`

One named experiment per module — keeps runs searchable and grouped from day one.

In [ ]:
EXPERIMENTS_PY = '''\
# src/mlflow_setup/experiments.py
# Central experiment registry — import EXPERIMENTS dict, never hardcode names.

from __future__ import annotations
import mlflow
from dataclasses import dataclass
from src.config.settings import settings


@dataclass(frozen=True)
class ExperimentSpec:
    name: str
    description: str
    tags: dict[str, str]


# ── One experiment per project module ─────────────────────────────────────
EXPERIMENTS: dict[str, ExperimentSpec] = {
    "infrastructure": ExperimentSpec(
        name="W1_Infrastructure",
        description="W1: env setup, scaffold, MLflow config",
        tags={"phase": "1", "week": "1", "module": "infrastructure"},
    ),
    "sku_intelligence": ExperimentSpec(
        name="W2_W4_SKU_Intelligence",
        description="W2-W4: data ingestion, SKU master, ABC/VED/FNS, Ci index",
        tags={"phase": "1", "week": "2-4", "module": "sku_intelligence"},
    ),
    "demand_forecasting": ExperimentSpec(
        name="W5_W7_Demand_Forecasting",
        description="W5-W7: ADI/CV² characterisation, Croston SBA, Holt-Winters, ARIMA",
        tags={"phase": "2", "week": "5-7", "module": "demand_forecasting"},
    ),
    "geo_risk_newsvendor": ExperimentSpec(
        name="W8_GeoRisk_Newsvendor",
        description="W8: Bayesian geo-risk scoring, Newsvendor Q*",
        tags={"phase": "2", "week": "8", "module": "geo_risk_newsvendor"},
    ),
    "supplier_qualification": ExperimentSpec(
        name="W9_W10_Supplier_Qualification",
        description="W9-W10: Decision tree supplier classifier, Nash equilibrium game",
        tags={"phase": "3", "week": "9-10", "module": "supplier_qualification"},
    ),
}


def setup_all_experiments(
    tracking_uri: str | None = None,
) -> dict[str, str]:
    """
    Create (or get) all project experiments in MLflow.
    Returns {key: experiment_id}.
    Safe to call repeatedly — MLflow is idempotent on existing experiments.
    """
    if tracking_uri:
        mlflow.set_tracking_uri(tracking_uri)

    client = mlflow.tracking.MlflowClient()
    ids: dict[str, str] = {}

    for key, spec in EXPERIMENTS.items():
        existing = client.get_experiment_by_name(spec.name)
        if existing is None:
            exp_id = client.create_experiment(
                name=spec.name,
                tags={"description": spec.description, **spec.tags},
            )
            status = "created"
        else:
            exp_id = existing.experiment_id
            status = "exists"
        ids[key] = exp_id
        print(f"  [{status:7s}] {spec.name:<40} id={exp_id}")

    return ids


def get_or_create(key: str, tracking_uri: str | None = None) -> mlflow.ActiveRun:
    """
    Convenience: set the experiment by key and return the active run context.

    Usage:
        with get_or_create("sku_intelligence"):
            mlflow.log_param("abc_cut_a", 0.80)
    """
    if tracking_uri:
        mlflow.set_tracking_uri(tracking_uri)
    spec = EXPERIMENTS[key]
    mlflow.set_experiment(spec.name)
    return mlflow.start_run()
'''

mlflow_dir = ROOT / "src" / "mlflow_setup"
mlflow_dir.mkdir(parents=True, exist_ok=True)
(mlflow_dir / "__init__.py").touch(exist_ok=True)
(mlflow_dir / "experiments.py").write_text(EXPERIMENTS_PY)
logger.info("✅ Written: src/mlflow_setup/experiments.py")

---
## STEP 3 — Create all experiments in the tracking server

In [ ]:
from src.mlflow_setup.experiments import setup_all_experiments, EXPERIMENTS

logger.info("🔧 Creating experiment taxonomy in MLflow...")
print()

experiment_ids = setup_all_experiments(tracking_uri=settings.MLFLOW_URI)

print(f"\n✅ {len(experiment_ids)} experiments registered")

---
## STEP 4 — Write `MLproject` file

`MLproject` is MLflow's project manifest — it defines:
- What conda env to use
- What entry points exist
- What parameters each entry point accepts

After this, `mlflow run . -e dummy_experiment` works from the command line.

In [ ]:
MLPROJECT = f"""\
name: geo-aware-mro

conda_env: environment.yml

entry_points:

  # ── W1: validation / smoke tests ────────────────────────────────────────
  dummy_experiment:
    parameters:
      experiment_name: {{type: str,   default: W1_Infrastructure}}
      n_skus:          {{type: int,   default: {settings.N_SKUS}}}
      random_seed:     {{type: int,   default: {settings.RANDOM_SEED}}}
    command: "python src/mlflow_setup/dummy_run.py
              --experiment_name {{experiment_name}}
              --n_skus {{n_skus}}
              --random_seed {{random_seed}}"

  # ── W3-W4: SKU Intelligence ──────────────────────────────────────────────
  sku_intelligence:
    parameters:
      w_abc:  {{type: float, default: {settings.W_ABC}}}
      w_ved:  {{type: float, default: {settings.W_VED}}}
      w_fns:  {{type: float, default: {settings.W_FNS}}}
      w_loc:  {{type: float, default: {settings.W_LOC}}}
    command: "python src/classifiers/run.py
              --w_abc {{w_abc}} --w_ved {{w_ved}}
              --w_fns {{w_fns}} --w_loc {{w_loc}}"

  # ── W5-W7: Demand Forecasting ────────────────────────────────────────────
  demand_forecasting:
    parameters:
      adi_threshold: {{type: float, default: 1.32}}
      cv2_threshold: {{type: float, default: 0.49}}
      forecast_horizon: {{type: int, default: 4}}
    command: "python src/forecasting/run.py
              --adi_threshold {{adi_threshold}}
              --cv2_threshold {{cv2_threshold}}
              --forecast_horizon {{forecast_horizon}}"

  # ── W8: GeoRisk + Newsvendor ─────────────────────────────────────────────
  geo_risk_newsvendor:
    parameters:
      w_hhi:   {{type: float, default: 0.40}}
      w_lpi:   {{type: float, default: 0.35}}
      w_gdelt: {{type: float, default: 0.25}}
    command: "python src/risk/run.py
              --w_hhi {{w_hhi}} --w_lpi {{w_lpi}} --w_gdelt {{w_gdelt}}"
"""

(ROOT / "MLproject").write_text(MLPROJECT)
logger.info("✅ Written: MLproject")
print()
print(MLPROJECT)

---
## STEP 5 — Write `src/mlflow_setup/dummy_run.py`

This is the **entry point** referenced in MLproject.
It simulates what every future module will do:
- Accept CLI args
- Log params, metrics, tags
- Save and log an artifact
- Register to the Model Registry

In [ ]:
DUMMY_RUN_PY = '''\
# src/mlflow_setup/dummy_run.py
# ── MLproject entry point: dummy_experiment ───────────────────────────────
# Simulates a minimal end-to-end MLflow run.
# Run via:
#   python src/mlflow_setup/dummy_run.py
#   mlflow run . -e dummy_experiment

from __future__ import annotations

import argparse
import json
import sys
import tempfile
from pathlib import Path
from datetime import datetime

import numpy as np
import mlflow
import mlflow.pyfunc

# ── Path bootstrap (works both as script and from notebook) ──────────────
try:
    ROOT = Path(__file__).resolve().parents[2]
except NameError:
    ROOT = Path.cwd()
    for p in [ROOT] + list(ROOT.parents):
        if (p / ".git").exists():
            ROOT = p
            break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config.settings import settings
from src.utils.logger import get_logger

logger = get_logger("DummyRun")


# ── Minimal sklearn-style model wrapper (pyfunc compatible) ──────────────
class DummySkuModel(mlflow.pyfunc.PythonModel):
    """
    Placeholder model wrapper.
    In W3-W4 this becomes the ABC/VED/FNS classifier.
    Registered today so the Model Registry has a v1 to build from.
    """
    def __init__(self, threshold: float = 0.5):
        self.threshold = threshold

    def predict(self, context, model_input):
        # Placeholder: returns "A"/"B"/"C" based on random scores
        rng = np.random.default_rng(42)
        scores = rng.random(len(model_input))
        return np.where(scores > 0.8, "A", np.where(scores > 0.5, "B", "C"))


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description="geo-mro dummy MLflow run")
    p.add_argument("--experiment_name", default="W1_Infrastructure")
    p.add_argument("--n_skus",          type=int,   default=settings.N_SKUS)
    p.add_argument("--random_seed",     type=int,   default=settings.RANDOM_SEED)
    return p.parse_args()


def simulate_sku_metrics(n_skus: int, seed: int) -> dict:
    """
    Generate fake but realistic SKU-level metrics.
    Mirrors what W3 will produce from the real M5 data.
    """
    rng = np.random.default_rng(seed)
    unit_costs     = rng.lognormal(mean=3.5, sigma=1.2, size=n_skus)
    demand_means   = rng.gamma(shape=2.0, scale=5.0, size=n_skus)
    annual_values  = unit_costs * demand_means * 52     # weekly demand → annual

    # ABC distribution
    sorted_vals = np.sort(annual_values)[::-1]
    cum_pct     = np.cumsum(sorted_vals) / sorted_vals.sum()
    n_a = int(np.searchsorted(cum_pct, 0.80)) + 1
    n_b = int(np.searchsorted(cum_pct, 0.95)) + 1 - n_a
    n_c = n_skus - n_a - n_b

    return {
        "n_skus":            n_skus,
        "n_class_a":         n_a,
        "n_class_b":         n_b,
        "n_class_c":         n_c,
        "pct_class_a":       round(n_a / n_skus, 4),
        "pct_class_b":       round(n_b / n_skus, 4),
        "pct_class_c":       round(n_c / n_skus, 4),
        "mean_unit_cost":    round(float(unit_costs.mean()), 2),
        "median_unit_cost":  round(float(np.median(unit_costs)), 2),
        "mean_annual_value": round(float(annual_values.mean()), 2),
        "total_annual_value":round(float(annual_values.sum()), 2),
        "gini_coefficient":  round(_gini(annual_values), 4),
    }


def _gini(x: np.ndarray) -> float:
    """Gini coefficient — measures concentration of value across SKUs."""
    x = np.sort(x)
    n = len(x)
    idx = np.arange(1, n + 1)
    return float((2 * (idx * x).sum()) / (n * x.sum()) - (n + 1) / n)


def run(args: argparse.Namespace) -> None:
    mlflow.set_tracking_uri(settings.MLFLOW_URI)
    mlflow.set_experiment(args.experiment_name)

    with mlflow.start_run(run_name="W1D2_dummy_run") as run:
        # ── 1. Log parameters ──────────────────────────────────────────────
        mlflow.log_params({
            "n_skus":          args.n_skus,
            "random_seed":     args.random_seed,
            "w_abc":           settings.W_ABC,
            "w_ved":           settings.W_VED,
            "w_fns":           settings.W_FNS,
            "w_loc":           settings.W_LOC,
            "adi_threshold":   1.32,
            "cv2_threshold":   0.49,
        })

        # ── 2. Simulate & log metrics ──────────────────────────────────────
        metrics = simulate_sku_metrics(args.n_skus, args.random_seed)
        mlflow.log_metrics(metrics)

        logger.info(f"  n_class_a={metrics[\'n_class_a\']}  "
                    f"n_class_b={metrics[\'n_class_b\']}  "
                    f"n_class_c={metrics[\'n_class_c\']}  "
                    f"gini={metrics[\'gini_coefficient\']}")

        # ── 3. Tags ────────────────────────────────────────────────────────
        mlflow.set_tags({
            "day":          "W1D2",
            "status":       "dummy_baseline",
            "data_source":  "synthetic_numpy",
            "model_type":   "placeholder_classifier",
            "run_date":     datetime.now().strftime("%Y-%m-%d"),
        })

        # ── 4. Log artifact: metrics summary JSON ──────────────────────────
        with tempfile.TemporaryDirectory() as tmpdir:
            artifact_path = Path(tmpdir) / "dummy_metrics_summary.json"
            artifact_path.write_text(json.dumps(metrics, indent=2))
            mlflow.log_artifact(str(artifact_path), artifact_path="reports")

        # ── 5. Log model + register ────────────────────────────────────────
        model = DummySkuModel(threshold=0.5)

        model_info = mlflow.pyfunc.log_model(
            artifact_path="sku_classifier",
            python_model=model,
            registered_model_name="SkuClassifier",
        )

        logger.info(f"✅ Run complete  | run_id={run.info.run_id}")
        logger.info(f"   Model URI     : {model_info.model_uri}")


if __name__ == "__main__":
    run(parse_args())
'''

(mlflow_dir / "dummy_run.py").write_text(DUMMY_RUN_PY)
logger.info("✅ Written: src/mlflow_setup/dummy_run.py")

---
## STEP 6 — Execute the dummy experiment run

This is the first **real** MLflow run. After this cell, the tracking server
has a logged experiment with params, metrics, artifacts, and a registered model.

In [ ]:
import mlflow
import mlflow.pyfunc
import json
import tempfile
import numpy as np
from datetime import datetime
from pathlib import Path

mlflow.set_tracking_uri(settings.MLFLOW_URI)
mlflow.set_experiment("W1_Infrastructure")


# ── Helper functions (inline — same logic as dummy_run.py) ───────────────
def _gini(x):
    x = np.sort(x)
    n = len(x)
    return float((2 * (np.arange(1, n+1) * x).sum()) / (n * x.sum()) - (n+1)/n)


def simulate_sku_metrics(n_skus, seed):
    rng = np.random.default_rng(seed)
    unit_costs    = rng.lognormal(mean=3.5, sigma=1.2, size=n_skus)
    demand_means  = rng.gamma(shape=2.0, scale=5.0, size=n_skus)
    annual_values = unit_costs * demand_means * 52
    sv = np.sort(annual_values)[::-1]
    cp = np.cumsum(sv) / sv.sum()
    n_a = int(np.searchsorted(cp, 0.80)) + 1
    n_b = int(np.searchsorted(cp, 0.95)) + 1 - n_a
    n_c = n_skus - n_a - n_b
    return {
        "n_skus":             n_skus,
        "n_class_a":          n_a,
        "n_class_b":          n_b,
        "n_class_c":          n_c,
        "pct_class_a":        round(n_a / n_skus, 4),
        "pct_class_b":        round(n_b / n_skus, 4),
        "pct_class_c":        round(n_c / n_skus, 4),
        "mean_unit_cost":     round(float(unit_costs.mean()), 2),
        "median_unit_cost":   round(float(np.median(unit_costs)), 2),
        "mean_annual_value":  round(float(annual_values.mean()), 2),
        "total_annual_value": round(float(annual_values.sum()), 2),
        "gini_coefficient":   round(_gini(annual_values), 4),
    }


# ── The dummy model ──────────────────────────────────────────────────────
class DummySkuModel(mlflow.pyfunc.PythonModel):
    """Placeholder. Replaced by real ABC/VED/FNS classifier in W3."""
    def __init__(self, threshold=0.5):
        self.threshold = threshold
    def predict(self, context, model_input):
        rng = np.random.default_rng(42)
        scores = rng.random(len(model_input))
        return np.where(scores > 0.8, "A", np.where(scores > 0.5, "B", "C"))


# ── Main run ─────────────────────────────────────────────────────────────
N_SKUS = settings.N_SKUS
SEED   = settings.RANDOM_SEED

with mlflow.start_run(run_name="W1D2_dummy_run") as run:

    # 1. Parameters
    mlflow.log_params({
        "n_skus":        N_SKUS,
        "random_seed":   SEED,
        "w_abc":         settings.W_ABC,
        "w_ved":         settings.W_VED,
        "w_fns":         settings.W_FNS,
        "w_loc":         settings.W_LOC,
        "adi_threshold": 1.32,
        "cv2_threshold": 0.49,
    })

    # 2. Metrics
    metrics = simulate_sku_metrics(N_SKUS, SEED)
    mlflow.log_metrics(metrics)

    # 3. Tags
    mlflow.set_tags({
        "day":         "W1D2",
        "status":      "dummy_baseline",
        "data_source": "synthetic_numpy",
        "model_type":  "placeholder_classifier",
        "run_date":    datetime.now().strftime("%Y-%m-%d"),
    })

    # 4. Artifact: JSON summary
    with tempfile.TemporaryDirectory() as tmpdir:
        art = Path(tmpdir) / "dummy_metrics_summary.json"
        art.write_text(json.dumps(metrics, indent=2))
        mlflow.log_artifact(str(art), artifact_path="reports")

    # 5. Log + register model
    model_info = mlflow.pyfunc.log_model(
        artifact_path="sku_classifier",
        python_model=DummySkuModel(threshold=0.5),
        registered_model_name="SkuClassifier",
    )

    RUN_ID = run.info.run_id

print("\n" + "═" * 55)
print("  DUMMY RUN COMPLETE")
print("═" * 55)
print(f"  run_id        : {RUN_ID}")
print(f"  experiment    : W1_Infrastructure")
print(f"  model_uri     : {model_info.model_uri}")
print()
print("  ABC distribution (simulated):")
print(f"    A: {metrics['n_class_a']:>4} SKUs ({metrics['pct_class_a']*100:.1f}%)")
print(f"    B: {metrics['n_class_b']:>4} SKUs ({metrics['pct_class_b']*100:.1f}%)")
print(f"    C: {metrics['n_class_c']:>4} SKUs ({metrics['pct_class_c']*100:.1f}%)")
print(f"  Gini coefficient : {metrics['gini_coefficient']}")
print("═" * 55)

---
## STEP 7 — Model Registry: transition to Staging

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# ── Get the latest version of SkuClassifier ───────────────────────────────
versions = client.get_latest_versions("SkuClassifier")
if versions:
    latest = versions[-1]
    version_num = latest.version

    # Transition from None → Staging
    client.transition_model_version_stage(
        name="SkuClassifier",
        version=version_num,
        stage="Staging",
        archive_existing_versions=False,
    )

    # Add description
    client.update_model_version(
        name="SkuClassifier",
        version=version_num,
        description=(
            "Placeholder SKU classifier — W1D2 dummy baseline. "
            "Replaced by ABC/VED/FNS real classifier in W3."
        ),
    )

    print(f"✅ SkuClassifier v{version_num} → Staging")
    print(f"   Source run : {latest.run_id}")
    print(f"   Created    : {latest.creation_timestamp}")
else:
    print("⚠️  No versions found — re-run Step 6")

---
## STEP 8 — Query the tracking server programmatically

This pattern is used in W7 to compare Croston vs Holt-Winters runs,
and in W8 to find the best-performing Newsvendor configuration.

In [ ]:
import pandas as pd

# ── Search all runs in W1_Infrastructure ─────────────────────────────────
runs_df = mlflow.search_runs(
    experiment_names=["W1_Infrastructure"],
    order_by=["start_time DESC"],
    max_results=20,
)

# ── Select the columns that matter ───────────────────────────────────────
cols = [
    "run_id",
    "tags.day",
    "tags.status",
    "params.n_skus",
    "params.random_seed",
    "metrics.n_class_a",
    "metrics.n_class_b",
    "metrics.n_class_c",
    "metrics.gini_coefficient",
    "metrics.total_annual_value",
    "start_time",
]
cols_present = [c for c in cols if c in runs_df.columns]
display_df = runs_df[cols_present].copy()
display_df["run_id"] = display_df["run_id"].str[:12] + "..."

print("\n📊 Runs in W1_Infrastructure:")
print(display_df.to_string(index=False))
print(f"\n✅ Total runs found: {len(runs_df)}")

---
## STEP 9 — Write `src/mlflow_setup/run_tracker.py`

A reusable context manager used by ALL future modules.
Every notebook from W2 onwards uses `with MroRun(...) as run:`.

In [ ]:
RUN_TRACKER_PY = '''\
# src/mlflow_setup/run_tracker.py
# ── Reusable MLflow run context manager ──────────────────────────────────
# Usage in any module:
#   from src.mlflow_setup.run_tracker import MroRun
#   with MroRun("sku_intelligence", run_name="W3D1_abc") as run:
#       mlflow.log_params({...})
#       mlflow.log_metrics({...})

from __future__ import annotations

import mlflow
import traceback
from contextlib import contextmanager
from typing import Generator

from src.config.settings import settings
from src.mlflow_setup.experiments import EXPERIMENTS
from src.utils.logger import get_logger

logger = get_logger("MroRun")


@contextmanager
def MroRun(
    experiment_key: str,
    run_name: str | None = None,
    tags: dict | None = None,
    tracking_uri: str | None = None,
) -> Generator[mlflow.ActiveRun, None, None]:
    """
    Context manager for all geo-mro MLflow runs.

    - Sets tracking URI from settings (override with tracking_uri)
    - Resolves experiment by key (from EXPERIMENTS dict)
    - On exception: sets run status to FAILED and re-raises
    - On success: sets run status to FINISHED

    Args:
        experiment_key: One of the keys in EXPERIMENTS
            ("infrastructure", "sku_intelligence", "demand_forecasting",
             "geo_risk_newsvendor", "supplier_qualification")
        run_name: Human-readable run name (e.g. "W3D1_abc_classifier")
        tags: Extra tags merged with defaults
        tracking_uri: Overrides settings.MLFLOW_URI

    Example:
        from src.mlflow_setup.run_tracker import MroRun

        with MroRun("sku_intelligence", run_name="W3D2_ved") as run:
            mlflow.log_param("density_threshold_V", 0.7)
            mlflow.log_metric("n_vital", 87)
    """
    uri = tracking_uri or settings.MLFLOW_URI
    mlflow.set_tracking_uri(uri)

    if experiment_key not in EXPERIMENTS:
        raise KeyError(
            f"Unknown experiment key: {experiment_key!r}. "
            f"Valid keys: {list(EXPERIMENTS)}"
        )

    spec = EXPERIMENTS[experiment_key]
    mlflow.set_experiment(spec.name)

    default_tags = {"project": settings.PROJECT_NAME, **spec.tags}
    merged_tags  = {**default_tags, **(tags or {})}

    with mlflow.start_run(run_name=run_name, tags=merged_tags) as active_run:
        logger.info(
            f"🚀 Run started | experiment={spec.name} | "
            f"run={run_name} | id={active_run.info.run_id[:12]}..."
        )
        try:
            yield active_run
        except Exception:
            mlflow.set_tag("run_status", "FAILED")
            logger.error(f"❌ Run FAILED\n{traceback.format_exc()}")
            raise
        else:
            mlflow.set_tag("run_status", "FINISHED")
            logger.info(
                f"✅ Run finished | id={active_run.info.run_id[:12]}..."
            )
'''

(mlflow_dir / "run_tracker.py").write_text(RUN_TRACKER_PY)
logger.info("✅ Written: src/mlflow_setup/run_tracker.py")

---
## STEP 10 — Smoke test `MroRun` context manager

In [ ]:
from src.mlflow_setup.run_tracker import MroRun

with MroRun("infrastructure", run_name="W1D2_mro_run_smoke_test") as run:
    mlflow.log_param("smoke_test", True)
    mlflow.log_metric("dummy_metric", 1.0)
    mlflow.set_tag("day", "W1D2")

print(f"\n✅ MroRun context manager working correctly")
print(f"   run_id: {run.info.run_id}")

---
## STEP 11 — Write pytest for MLflow setup

Every module written gets a test. CI will run this on every push.

In [ ]:
TEST_MLFLOW_PY = '''\
# tests/unit/test_mlflow_setup.py

import pytest
import mlflow
from unittest.mock import patch

from src.config.settings import settings
from src.mlflow_setup.experiments import EXPERIMENTS, setup_all_experiments
from src.mlflow_setup.run_tracker import MroRun


# ── Fixtures ──────────────────────────────────────────────────────────────
@pytest.fixture(autouse=True)
def use_sqlite_backend(tmp_path):
    """Redirect MLflow to a temp SQLite DB for every test — no side effects."""
    uri = f"sqlite:///{tmp_path / \'test_mlflow.db\'}"
    mlflow.set_tracking_uri(uri)
    yield
    mlflow.end_run()    # safety: close any leaked run


# ── Tests ─────────────────────────────────────────────────────────────────
class TestExperiments:
    def test_experiments_dict_has_five_keys(self):
        assert len(EXPERIMENTS) == 5

    def test_all_experiments_have_required_fields(self):
        for key, spec in EXPERIMENTS.items():
            assert spec.name, f"{key} has no name"
            assert spec.description, f"{key} has no description"
            assert "phase" in spec.tags, f"{key} tags missing phase"

    def test_setup_all_experiments_creates_entries(self):
        ids = setup_all_experiments()
        assert len(ids) == 5
        for key, exp_id in ids.items():
            assert exp_id is not None

    def test_setup_idempotent(self):
        """Calling setup_all_experiments twice must not raise."""
        ids1 = setup_all_experiments()
        ids2 = setup_all_experiments()
        assert ids1 == ids2


class TestMroRun:
    def test_valid_key_logs_run(self):
        with MroRun("infrastructure", run_name="test_run") as run:
            mlflow.log_param("x", 1)
        assert run.info.run_id is not None

    def test_invalid_key_raises(self):
        with pytest.raises(KeyError):
            with MroRun("nonexistent_key"):
                pass

    def test_exception_sets_failed_tag(self):
        with pytest.raises(ValueError):
            with MroRun("infrastructure", run_name="fail_run") as run:
                raise ValueError("intentional failure")
        # After the run, check the tag
        client = mlflow.tracking.MlflowClient()
        tags = client.get_run(run.info.run_id).data.tags
        assert tags.get("run_status") == "FAILED"

    def test_success_sets_finished_tag(self):
        with MroRun("infrastructure", run_name="success_run") as run:
            mlflow.log_metric("dummy", 1.0)
        client = mlflow.tracking.MlflowClient()
        tags = client.get_run(run.info.run_id).data.tags
        assert tags.get("run_status") == "FINISHED"


class TestSettings:
    def test_weights_sum_to_one(self):
        total = settings.W_ABC + settings.W_VED + settings.W_FNS + settings.W_LOC
        assert abs(total - 1.0) < 1e-9, f"Weights sum to {total}, not 1.0"

    def test_n_skus_positive(self):
        assert settings.N_SKUS > 0

    def test_mlflow_uri_is_sqlite(self):
        assert settings.MLFLOW_URI.startswith("sqlite:///")
'''

test_unit_dir = ROOT / "tests" / "unit"
test_unit_dir.mkdir(parents=True, exist_ok=True)
(test_unit_dir / "__init__.py").touch(exist_ok=True)
(test_unit_dir / "test_mlflow_setup.py").write_text(TEST_MLFLOW_PY)
logger.info("✅ Written: tests/unit/test_mlflow_setup.py")

---
## STEP 12 — Run pytest

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pytest",
     "tests/unit/test_mlflow_setup.py",
     "-v", "--tb=short", "--no-header",
     f"--rootdir={ROOT}"],
    cwd=ROOT,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("❌ Tests failed — fix before committing")
else:
    print("\n✅ All tests passed")

---
## STEP 13 — MLflow UI launch instructions

Run this in a **separate terminal tab** — it stays running while you work.

In [ ]:
DB_PATH = settings.MLFLOW_DIR / "mlflow.db"

UI_COMMANDS = f"""
╔══════════════════════════════════════════════════════════════╗
║  LAUNCH MLflow UI (run in a new terminal tab)               ║
╚══════════════════════════════════════════════════════════════╝

conda activate geo-mro

mlflow server \\
    --backend-store-uri sqlite:///{DB_PATH} \\
    --default-artifact-root {settings.MLFLOW_DIR} \\
    --host 0.0.0.0 \\
    --port 5000

Then open:  http://localhost:5000

You should see:
  ► Experiments: W1_Infrastructure (+ 4 others)
  ► Runs: W1D2_dummy_run, W1D2_mro_run_smoke_test
  ► Models: SkuClassifier v1 (Staging)
"""
print(UI_COMMANDS)

---
## STEP 14 — Commit Day 2 artefacts

In [ ]:
GIT_COMMANDS = f"""
# ── Run in terminal from {ROOT} ────────────────────────────────────────────

cd "{ROOT}"

git checkout develop              # all Day 2 work goes on develop

git add MLproject \\
        src/mlflow_setup/ \\
        tests/unit/test_mlflow_setup.py

git commit -m "feat: W1D2 — MLflow tracking server, MLproject, experiment taxonomy, MroRun context manager"

git push origin develop
"""
print(GIT_COMMANDS)

---
## ✅ Day 2 Checklist

| Task | Done? |
|------|-------|
| Day 1 gates passed | ☐ |
| MLflow tracking URI set to SQLite backend | ☐ |
| 5 named experiments created in MLflow | ☐ |
| `MLproject` YAML written (4 entry points) | ☐ |
| `src/mlflow_setup/experiments.py` written | ☐ |
| `src/mlflow_setup/dummy_run.py` written | ☐ |
| `src/mlflow_setup/run_tracker.py` written | ☐ |
| Dummy experiment run logged (params + metrics + artifact) | ☐ |
| `SkuClassifier` v1 registered → Staging | ☐ |
| `MroRun` context manager smoke-tested | ☐ |
| Pytest: 8 tests all green | ☐ |
| MLflow UI visible at localhost:5000 | ☐ |
| Committed to `develop` + pushed | ☐ |

---

## 🔜 Day 3 (Wed) Preview

**DVC init, remote storage, data directory versioning**

- `dvc init` + `.dvcignore`
- Configure local remote (Day 3) → S3 remote (Day 4 optional)
- `dvc add data/raw/` — start tracking M5 dataset files
- First `dvc push` / `dvc pull` cycle
- DVC pipeline YAML (`dvc.yaml`) stub for W2 ingestion